In [3]:
from google.colab import drive
drive.mount('/content/drive')
import matplotlib.pyplot as plt
import numpy as np
import time

from numpy.polynomial.legendre import legval
from scipy.special import eval_legendre,factorial
from scipy.optimize import least_squares

Mounted at /content/drive


# Import des données

In [4]:
# Tableau : He-HCN
He = np.genfromtxt("/content/drive/MyDrive/Master/Stage/IPR/He-HCN.txt",
                   delimiter=",", names=True, dtype=None, encoding="utf-8")

# Optimisation des fonctions

Dans ce Notebook, on s'intéresse à l'optimisation des performances dans l'implémentation de notre fonction :
$$V(R,\theta)=V_{sh}(R,\theta)+V_{as}(R,\theta)$$

Il me semble important de passer un peu de temps dessus (c'est peut être une erreur de ma part) car ces fonctions seront appelées de nombreuses fois pour l'estimation des paramètres optimaux.

## Premier jet

In [5]:
def G(R,Theta,g0,g1,g2,g3):
  g = 0.
  for i in range(6):
    g = g + (g0[i]+g1[i]*R+g2[i]*R*R+g3[i]*R*R*R)*eval_legendre(i,np.cos(Theta))
  return g

In [6]:
def X(Theta,x):
  y = 0.
  for i in range(6):
    y = y + x[i]*eval_legendre(i,np.cos(Theta))
  return y

In [7]:
def f(n,x):
  y = 0.
  for k in range(n+1):
    y = y + (x**k)/factorial(k)
  y = 1. - np.exp(-x)*y
  return y

In [8]:
def V(vars, *params):

  R, Theta = vars

  # Conversion des degrés en radian
  Theta_rad = np.deg2rad(Theta)

  #  Récupération des paramètres
  b  = params[0:6]    # b0, b1, b2, b3, b4, b5
  c  = params[6:10]   # c0, c1, c2, c3
  d  = params[10:16]  # d0, d1, d2, d3, d4, d5
  g0 = params[16:22]  # g00, g01, g02, g03, g04, g05
  g1 = params[22:28]  # g10, g11, g12, g13, g14, g15
  g2 = params[28:34]  # g20, g21, g22, g23, g24, g25
  g3 = params[34:40]  # g30, g31, g32, g33, g34, g35

  # Variables utiles
  abs_BR = np.abs(X(Theta_rad,b)*R)
  cosTh = np.cos(Theta_rad)

  # Short-Range
  V_sh = G(R,Theta_rad,g0,g1,g2,g3) * np.exp(X(Theta_rad,d)-X(Theta_rad,b)*R)

  # Asymptotyhic
  V_as = f(6,abs_BR) * (c[0]*eval_legendre(0,cosTh)+c[2]*eval_legendre(2,cosTh))/(R**6)\
       + f(7,abs_BR) * (c[1]*eval_legendre(1,cosTh)+c[3]*eval_legendre(3,cosTh))/(R**7)

  return V_sh + V_as

## Versions optimisées

**Méthodologie :**

Après avoir implémenté les différentes fonctions qui sont présentées dans l'article "Theoretical study of the He-HCN,..., complexes".

Je me sert de différents LLM pour trouver des pistes d'améliorations des performances de mon premier programme.

Avec des prompts du style "Quel est le plus performant entre telle chose et telle chose" ou "Comment puis je améliorer les performances" etc.

Liste des points à améliorer :

1.   Certaines variables sont recalculées plusieurs fois de manière inutile
2.   Vectorisation de certaines fonctions, éviter l'appel de fonctions exterieur lorsque ce n'est pas nécessaire
3.   Utilisation de fonctions numpy comme legval lorsque c'est utile



In [9]:
def f6_opt(x):
  y = 1.0 + x * (1.0 + x * (0.5 + x * (1.0/6.0 + x * (1.0/24.0 + x * (1.0/120.0 + x / 720.0)))))
  return 1.0 - np.exp(-x) * y

def f7_opt(x):
  y = 1.0 + x * (1.0 + x * (0.5 + x * (1.0/6.0 + x * (1.0/24.0 + x * (1.0/120.0 + x * (1.0/720.0 + x / 5040.0))))))
  return 1.0 - np.exp(-x) * y

In [10]:
def V_opt(p, R, Theta):
  R = np.asarray(R, dtype=float)
  # On calcule ces valeurs car utiles plusieurs fois
  cosTh = np.cos(np.deg2rad(np.asarray(Theta, dtype=float)))

  # Paramètres
  b  = p[0:6]                                        # b0, b1, b2, b3, b4, b5
  c  = p[6:10]                                       # c0, c1, c2, c3
  d  = p[10:16]                                      # d0, d1, d2, d3, d4, d5
  g0 = np.asarray(p[16:22], dtype=float).reshape(6,) # g00, g01, g02, g03, g04, g05
  g1 = np.asarray(p[22:28], dtype=float).reshape(6,) # g10, g11, g12, g13, g14, g15
  g2 = np.asarray(p[28:34], dtype=float).reshape(6,) # g20, g21, g22, g23, g24, g25
  g3 = np.asarray(p[34:40], dtype=float).reshape(6,) # g30, g31, g32, g33, g34, g35

  # X(theta) ------------------------------------ #
  # --------------------------------------------- #
  X_b = legval(cosTh, b)
  X_d = legval(cosTh, d)

  X_bR = X_b * R
  abs_BR = np.abs(X_bR)
  # --------------------------------------------- #

  # G(R,theta) ---------------------------------- #
  # --------------------------------------------- #
  i = np.arange(6)
  g = g0[i, None] + R * (g1[i, None] + R * (g2[i, None] + R * g3[i, None]))
  Pl = np.array([eval_legendre(l, cosTh) for l in range(6)])
  # G = np.sum(g * Pl, axis=0)
  # ------------------------------------------- #

  # Short-Range
  V_sh = np.sum(g * Pl, axis=0) * np.exp(X_d-X_bR)

  # Asymptotic
  inv_R = 1.0/R
  inv_R6 = inv_R * inv_R * inv_R * inv_R * inv_R * inv_R
  inv_R7 = inv_R6 * inv_R

  V_as = (f6_opt(abs_BR) * legval(cosTh,[c[0],0,c[2]]) * inv_R6
        + f7_opt(abs_BR) * legval(cosTh,[0,c[1],0,c[3]]) * inv_R7)

  return V_sh + V_as

In [11]:
np.random.seed(0)

R = np.random.uniform(2.0, 10.0, 500)
Theta = np.random.uniform(0.0, 180.0, 500)

p = np.random.randn(40)

V1 = V((R, Theta), *p)
V2 = V_opt(p, R, Theta)

diff = V1 - V2

print("max diff :", np.max(np.abs(diff)))
print("mean diff:", np.mean(np.abs(diff)))
print("std diff :", np.std(diff))

max diff : 1.7881393432617188e-06
mean diff: 1.2293582947190762e-08
std diff : 1.0907050426399723e-07


# Tests de performances

In [12]:
def residus_1(params, R, Theta, V_tableau):
    return  V((R, Theta), *params) - V_tableau

def residus_2(p, R,Theta, V_tableau):
    return V_opt(p, R, Theta) - V_tableau

In [13]:
R = He["R"]
Theta = He["Theta"]
V_tableau = He["Energy"]

params_init = np.random.rand(40)
params_init

array([4.75324782e-01, 9.69205872e-01, 2.65632548e-01, 1.35087066e-02,
       4.83752865e-01, 2.56113795e-01, 8.23717672e-01, 2.32772672e-01,
       3.10629218e-01, 7.91227431e-01, 7.15143252e-01, 5.58051237e-01,
       7.04948062e-01, 4.18636864e-01, 5.31004761e-03, 1.13551285e-02,
       5.11221788e-01, 8.32909797e-02, 5.10754802e-02, 9.65516639e-01,
       8.59002640e-01, 1.52027227e-01, 6.64218590e-04, 9.41667795e-01,
       2.78325298e-01, 1.85897603e-01, 6.91508108e-01, 1.08903739e-01,
       2.64649598e-01, 9.75094680e-01, 6.39462774e-01, 5.20677791e-01,
       3.97918615e-01, 7.74500955e-01, 1.40957477e-01, 9.67337802e-01,
       8.61123008e-01, 6.17656983e-01, 4.29061904e-02, 7.00855649e-01])

In [14]:
# On appelle les deux fonctions une première fois pour retirer les couts liés à
# l'initialisation des fonctions
residus_1(params_init, R, Theta, V_tableau);
residus_2(params_init, R, Theta, V_tableau);

start = time.perf_counter()
resol1 = least_squares(residus_1, params_init, args=(R, Theta, V_tableau))
end = time.perf_counter()
temps1 = end - start

start = time.perf_counter()
resol2 = least_squares(residus_2, params_init, args=(R, Theta, V_tableau))
end = time.perf_counter()
temps2 = end - start

print(f"Temps sans optimisation : {temps1}s")
print(f"Temps avec optimisation : {temps2}s")

err_relative = np.abs((resol1.x - resol2.x) / resol1.x)
print("Erreur maximale relative :", np.max(err_relative))
print("Erreur relative moyenne :", np.mean(err_relative))

Temps sans optimisation : 14.681737256999895s
Temps avec optimisation : 6.312615463000384s
Erreur maximale relative : 0.2068828825751537
Erreur relative moyenne : 0.005351842071192372


In [21]:
y = X(np.radians(np.linspace(0,180,18)),[1.2311200436701377, 0.01867704647616706, -0.09840359879698286, 0.017070376266523565, 0.036682970412129666, 0.013602357658515675])
y

array([1.2187492 , 1.21242634, 1.19839843, 1.18826347, 1.19273245,
       1.21468131, 1.2471592 , 1.27730483, 1.29331655, 1.29000712,
       1.26993829, 1.24035187, 1.20868156, 1.1796802 , 1.15533899,
       1.13652038, 1.12432027, 1.12004963])

In [22]:
y = X(np.radians(np.linspace(0,180,18)),[10.016402160493948, 0.9623076188371255, 0.031168234761964526, 0.2456494750926912, 0.32321900777673257, 0.11175023441652554])
y

array([11.69049673, 11.56813594, 11.2525246 , 10.8690531 , 10.5482575 ,
       10.35980946, 10.2876837 , 10.25643218, 10.18605215, 10.0383614 ,
        9.82893576,  9.60495058,  9.41083733,  9.26648793,  9.16808057,
        9.10322696,  9.0643786 ,  9.05108207])